# 0️⃣7️⃣ - How to use SNMP and NETCONF
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Intermediate-yellow) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to perform SNMP walk and get operations, and execute NETCONF/YANG calls against devices.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to identify SNMP-enabled devices in your RADKit inventory |
| 2 | How to perform an SNMP walk against a device |
| 3 | How to identify NETCONF-enabled devices in your RADKit inventory |
| 4 | How to access and navigate YANG modules on a device |
| 5 | How to query a YANG node using `get()` |
| 6 | How to query one or more explicit XPaths using `get_xpaths()` |

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

---

### 🤖 The `snmp` and `netconf` Objects

When a device is onboarded in RADKit with SNMP or NETCONF enabled, the SDK exposes dedicated sub-objects on each `Device` instance:

- **`device.snmp`**: entry point for all SNMP operations (walk, get) against that device
- **`device.netconf`**: entry point for all NETCONF/YANG operations, including capability discovery and data retrieval

Both objects are only available when the corresponding protocol has been configured in the device inventory. The examples below show how to detect which devices support each protocol and how to interact with them.

---

## 📡 Method 1: SNMP

**Best for:** Querying device health, interface stats, and MIB data on devices where SNMP is already running: no agent-side changes required beyond enabling it in the RADKit inventory.

**How it works:**
1. Enable SNMP on the device entry in RADKit (WebUI or API), setting version, port, and community string.
2. Use `device.snmp.walk()` or `device.snmp.get()` with an OID to retrieve MIB data through the RADKit service.

**What you need:**
- A device with SNMP configured in the RADKit inventory
- The OID you want to query

### 1.1 Discovering SNMP-enabled devices

You can fetch SNMP-capable devices from your inventory by checking for the presence of the `snmp_version` internal attribute:

In [6]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    for dev in service.inventory.values():
        try:
            if dev.attributes.internal["snmp_version"]:
                print(f"📱 Device {dev.name} supports SNMP ...")
        except KeyError:
            continue


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
📱 Device p0-2e supports SNMP ...


---

### 1.2 Performing an SNMP walk

RADKit allows you to perform an SNMP walk against a target device using a specified OID. The call is asynchronous; chain `.wait()` to block until the result is ready:

In [ ]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    sys_oid = service.inventory['p0-2e'].snmp.walk("1.3.6.1.2.1.1").wait()
    print(sys_oid)

---

## 🧱 Method 2: NETCONF

**Best for:** Structured configuration and state data retrieval using YANG models — ideal when you need precise, schema-validated data rather than raw CLI or MIB output.

**How it works:**
1. Enable NETCONF on the device entry in RADKit (WebUI or API), providing the port and credentials.
2. Call `device.update_netconf().wait()` to pull the device's NETCONF capabilities.
3. Navigate the YANG tree via `device.netconf.yang` or query specific XPaths with `device.netconf.get_xpaths()`.

**Requirements:**

| Requirement | Details |
|---|---|
| NETCONF enabled on device | The device must support NETCONF (typically port 830) |
| RADKit inventory configured | Tick the NETCONF box in the device entry and supply port + credentials |
| Capabilities fetched | Call `update_netconf().wait()` before accessing YANG models |

### 2.1 Discovering NETCONF-enabled devices

Use an inventory filter against the `netconf_config` attribute to list only NETCONF-capable devices:

In [ ]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    nc_devices = service.inventory.filter("netconf_config", "True")
    for device_name in nc_devices.values():
        print(f"📱 Device {device_name.name} supports NETCONF ...")    


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
📱 Device p0-2e supports NETCONF ...


---

### 2.2 Accessing YANG modules

To get started, call `update_netconf().wait()` to pull the capabilities available on the device. Once this is done, the YANG tree is accessible via `device.netconf.yang`.

In this example, we update the capabilities and access different attributes of the `ietf-interfaces` model. The resulting object is an instance of the `SingleDeviceYangNode` class:

In [ ]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    nc_device = service.inventory['p0-2e']
    nc_device.update_netconf().wait()
    print(f"🧱 NETCONF capabilities now updated!")
    
    interfaces_yang_container = nc_device.netconf.yang['ietf-interfaces']['interfaces']
    print(f"🚸 XPath of this container is {interfaces_yang_container.xpath}")
    print(f"🕸️ Structure (default print): {interfaces_yang_container}")
    print(f"📚 Namespaces of this container are {interfaces_yang_container.namespaces}")
    


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
🧱 NETCONF capabilities now updated!
🚸 XPath of this container is /ietf-interfaces:interfaces
🕸️ Structure (default print): SingleDeviceYangNode(device_name='p0-2e', xpath='/ietf-interfaces:interfaces')

{
    "interface": {
        "name": {},
        "description": {},
        "type": {},
        "enabled": {},
        "link-up-down-trap-enable": {}
    }
}
📚 Namespaces of this container are {'ietf-interfaces': 'urn:ietf:params:xml:ns:yang:ietf-interfaces'}


---

### 2.3 Querying a YANG node

Use `get()` to retrieve the contents at the selected YANG path. This returns a `TransformedFillerRequest` representing the actual NETCONF request sent to the device. Access `.result.yang` to read the returned data:

In [ ]:
import json
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    nc_device = service.inventory['p0-2e']
    nc_device.update_netconf().wait()
    print(f"🧱 NETCONF capabilities now updated!")
    
    interfaces_yang_container = nc_device.netconf.yang['ietf-interfaces']['interfaces']['interface']
    interfaces_dict = dict(interfaces_yang_container.get().wait().result.yang)
    
    print(f"🕸️ Interface FortyGigabitEthernet1/1/1 data (using dictionary structure): \n\n{json.dumps(interfaces_dict["interfaces"]["interface"]["FortyGigabitEthernet1/1/1"], indent=2)}")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
🧱 NETCONF capabilities now updated!
🕸️ Interface FortyGigabitEthernet1/1/1 data (using dictionary structure): 

{
  "name": "FortyGigabitEthernet1/1/1",
  "type": {
    "@xmlns:ianaift": "urn:ietf:params:xml:ns:yang:iana-if-type",
    "#text": "ianaift:ethernetCsmacd"
  },
  "enabled": "true",
  "ipv4": {
    "@xmlns": "urn:ietf:params:xml:ns:yang:ietf-ip"
  },
  "ipv6": {
    "@xmlns": "urn:ietf:params:xml:ns:yang:ietf-ip"
  }
}


---

### 2.4 Querying explicit XPaths

If you know the exact XPath you need, you can query it directly with `get_xpaths()`. You can also pass multiple XPaths in a single request, which is more efficient than separate calls.

This method returns a `XPathsResult` instance. Each item in the result is a tuple containing the queried XPath and its corresponding YANG data:

In [ ]:
import json
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    nc_device = service.inventory['p0-2e']
    nc_device.update_netconf().wait()
    print(f"🧱 NETCONF capabilities now updated!")
    
    yang_request = nc_device.netconf.get_xpaths([
        "/ietf-interfaces:interfaces/interface[name='FortyGigabitEthernet1/1/1']/ipv4",
        "/openconfig-system:system/state/current-datetime"
    ]).wait()
    
    for item in yang_request.result.items():
        print(f"\nXPath: {item[0]}")
        print(f"YANG Data: {json.dumps(dict(item[1].yang), indent=2)}\n--------------")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
🧱 NETCONF capabilities now updated!

XPath: /ietf-interfaces:interfaces/interface[name='FortyGigabitEthernet1/1/1']/ipv4
YANG Data: {
  "interfaces": {
    "@xmlns": "urn:ietf:params:xml:ns:yang:ietf-interfaces",
    "interface": {
      "name": "FortyGigabitEthernet1/1/1",
      "ipv4": {
        "@xmlns": "urn:ietf:params:xml:ns:yang:ietf-ip"
      }
    }
  }
}
--------------

XPath: /openconfig-system:system/state/current-datetime
YANG Data: {
  "system": {
    "@xmlns": "http://openconfig.net/yang/system",
    "state": {
      "current-datetime": "2026-04-02T15:10:44Z+00:00"
    }
  }
}
--------------


---

## 🔀 Which Method Should I Use?

| | 📡 SNMP | 🧱 NETCONF |
|---|---|---|
| **Data model** | MIB/OID-based (flat) | YANG-based (hierarchical, schema-validated) |
| **Use case** | Polling metrics, interface stats, device health | Structured config and state retrieval |
| **RADKit setup** | Enable SNMP on device entry (version, port, community) | Enable NETCONF on device entry (port, credentials) |
| **First call** | `device.snmp.walk()` or `device.snmp.get()` | `device.update_netconf().wait()`, then navigate YANG tree |
| **Best for** | Quick read-only polling of standard MIBs | Precise, vendor-agnostic config/state data via YANG |